# MSSearch Next-Gen Redesign: Comprehensive Multi-Sample Evaluation Suite (N=30)
## Quantitative Benchmark & Multi-Regime Comparative Diagnostic against Old CVSearch Baseline

This notebook executes an end-to-end multi-sample benchmark across **$N=30$ target samples** from the high-resolution visual search suite (HRBench-8k/4k & V*), comparing four distinct operational regimes:
1. **Direct Full-Canvas Baseline (Old Stage 2)**: Standard VLM direct question answering over downsampled full canvases ($1024\text{px}$).
2. **Old CVSearch Quadtree Baseline**: Rigid $2\times 2$ bisection quadtree, global average-pooled cosine similarity saliency, binary pruning cliff gating ($\tau_{\text{prune}} = 0.25$), and geometric centroid reading.
3. **Redesign Stage 4 Macro Holistic Context**: Elastic Sobel seam-carving, dense spatial token saliency $S^*(p)$, continuous Bayesian evidential consolidation ($E_{\text{post}} = E_{\text{prior}} + \beta c_{\text{VLM}}$), and unified connected-component macro-envelopes.
4. **Redesign Stage 6 Foveated Native 1:1 Peak Detail**: Unscaled native optical resolution crop centered on the Argmax Evidential Peak $p^*$ (eliminating micro-object dilution).
5. **Redesign Unified Engine (Ensemble/Consensus)**: Adaptive selection combining Macro Context and Native 1:1 Peak Detail.

### Artifacts & Deliverables
- **Accuracies**: Global accuracy (%) across all regimes, plus fine-grained subcategory breakdowns (Attribute / Direct vs. Relational / Spatial).
- **Per-Sample Visual Cards**: High-resolution 4-panel visual diagnostic cards saved for every evaluated sample to `outputs/eval_multisource_n30/visualizations/sample_{idx:03d}_card.png`.
- **Global Summary Visualizations**: Publication-grade comparative bar charts, pruning cliff diagnostic plots, and latency Pareto frontiers saved to `outputs/eval_multisource_n30/`.
- **Telemetry Checkpoint**: Streaming JSONL telemetry logs saved to `outputs/eval_multisource_n30/telemetry/`.

In [ ]:
# ==============================================================================
# CELL 1: ENVIRONMENT SETUP, GITHUB REPO SYNC & HARDWARE TELEMETRY
# ==============================================================================
import os
import sys
import gc
import json
import re
import time
import shutil
import random
import warnings
import subprocess
import importlib
import urllib.request
from typing import List, Dict, Tuple, Optional, Any, Set, Union
from dataclasses import dataclass, field

# Enforce strict offline operation for reproducible Kaggle runs
os.environ["TOKENIZERS_PARALLELISM"] = "false"

warnings.filterwarnings("ignore")

import torch
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# ------------------------------------------------------------------------------
# 1. Automatic MSSearch Repository Discovery or Authenticated Clone
# ------------------------------------------------------------------------------
REPO_DIR = "/kaggle/working/MSSearch"
github_token = None

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    github_token = user_secrets.get_secret("MaCVi")
    print("[SETUP] Successfully retrieved GitHub token from Kaggle User Secret 'MaCVi'.")
except Exception:
    github_token = os.environ.get("MaCVi") or os.environ.get("GITHUB_TOKEN")

if not os.path.exists(REPO_DIR):
    local_candidates = [
        os.path.abspath("MSSearch"),
        os.path.abspath("../MSSearch"),
        os.path.abspath("../../MSSearch"),
        os.path.abspath("."),
        os.path.abspath(".."),
        "/kaggle/working/mysite/docs/experiments/multisource/MSSearch"
    ]
    found_local = next((c for c in local_candidates if os.path.exists(os.path.join(c, "mssearch", "__init__.py"))), None)
    
    if found_local:
        REPO_DIR = found_local
        print(f"[SETUP] Discovered existing local MSSearch package at: {REPO_DIR}")
    else:
        if github_token:
            print(f"[SETUP] Cloning MSSearch repository using authenticated token into {REPO_DIR}...")
            clone_url = f"https://{github_token}@github.com/tothanhdat2006/MSSearch.git"
        else:
            print(f"[SETUP] Attempting clone of MSSearch repository into {REPO_DIR}...")
            clone_url = "https://github.com/tothanhdat2006/MSSearch.git"
        ret = subprocess.run(["git", "clone", clone_url, REPO_DIR], capture_output=True, text=True)
        if ret.returncode == 0:
            print(f"[SETUP] Successfully cloned MSSearch repository into {REPO_DIR}")
        else:
            safe_err = ret.stderr.replace(github_token, "******") if github_token else ret.stderr
            print(f"[SETUP] Git clone failed: {safe_err}")
else:
    print(f"[SETUP] MSSearch repository already exists at {REPO_DIR}. Pulling latest changes from main...")
    ret = subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], capture_output=True, text=True)
    if ret.returncode == 0:
        print(f"[SETUP] Repository updated to latest commit: {ret.stdout.strip()}")

# Ensure REPO_DIR is at the front of sys.path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
elif sys.path[0] != REPO_DIR:
    sys.path.remove(REPO_DIR)
    sys.path.insert(0, REPO_DIR)

importlib.invalidate_caches()
for mod in list(sys.modules.keys()):
    if mod.startswith("mssearch"):
        del sys.modules[mod]

# Ensure BPE vocab file is present for SAM3 tokenizer
bpe_target = os.path.join(REPO_DIR, "assets", "bpe_simple_vocab_16e6.txt.gz")
if not os.path.exists(bpe_target):
    os.makedirs(os.path.dirname(bpe_target), exist_ok=True)
    bpe_url = "https://raw.githubusercontent.com/openai/CLIP/main/clip/bpe_simple_vocab_16e6.txt.gz"
    print(f"[SETUP] Downloading missing BPE vocab to {bpe_target}...")
    try:
        urllib.request.urlretrieve(bpe_url, bpe_target)
        print("[SETUP] Successfully downloaded BPE vocab asset.")
    except Exception as e:
        print(f"[SETUP WARNING] BPE download failed: {e}")

# Install dependencies quietly
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.45.0", "accelerate>=0.26.0", "tabulate", "iopath", "decord", "ftfy", "scikit-image", "scikit-learn", "networkx"], check=False)

# Hardware Telemetry
device_count = torch.cuda.device_count()
print("=" * 75)
print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
print(f"GPU Device Count: {device_count}")
for i in range(device_count):
    prop = torch.cuda.get_device_properties(i)
    free_mem, total_mem = torch.cuda.mem_get_info(i)
    print(f"  [GPU {i}] {prop.name} | Total: {total_mem/(1024**3):.2f} GiB | Free: {free_mem/(1024**3):.2f} GiB")
device_vlm = torch.device("cuda:0") if device_count > 0 else torch.device("cpu")
device_sam = torch.device("cuda:1") if device_count > 1 else device_vlm
print(f"[Device Topology] VLM & SigLIP Device: {device_vlm} | SAM3 Device: {device_sam}")
print("=" * 75)


In [ ]:
# ==============================================================================
# CELL 2: DATASET DISCOVERY & MULTI-SAMPLE (N=30) CANDIDATE POOL SUITE
# ==============================================================================
HRBENCH_SEARCH_DIRS = [
    "D:/Research/VisualSearch/hr-bench_8k",
    "d:/Research/VisualSearch/hr-bench_8k",
    "/kaggle/input/datasets/thnhtt/vs-dataset/hr-bench_8k/mnt/sdb/llp/datasets/zoom_eye_data/zoom_eye_data/hr-bench_8k",
    "/kaggle/input/datasets/thnhtt/vs-dataset/hr-bench_8k",
    "/kaggle/input/datasets/thnhtt/vs-dataset/hr-bench_4k",
    "/kaggle/input/hr-bench-8k",
    "/kaggle/input/hr-bench-4k",
    "docs/experiments/multisource/data/hr-bench_8k",
    "docs/experiments/freeform/ICML26-CVSearch/data/hr-bench_8k"
]

HRBENCH_BASE_DIR = None
for d in HRBENCH_SEARCH_DIRS:
    if os.path.exists(d):
        HRBENCH_BASE_DIR = d
        break

if HRBENCH_BASE_DIR is None:
    raise FileNotFoundError(f"[Dataset Error] Could not find HRBench directory in candidate paths: {HRBENCH_SEARCH_DIRS}")

HRBENCH_ANNOTATION_PATH = os.path.join(HRBENCH_BASE_DIR, "annotation_hr-bench_8k.json")
if not os.path.exists(HRBENCH_ANNOTATION_PATH):
    HRBENCH_ANNOTATION_PATH = os.path.join(HRBENCH_BASE_DIR, "annotation_hr-bench_4k.json")

print(f"[Dataset Discovery] Using HRBench root: {HRBENCH_BASE_DIR}")
print(f"[Dataset Discovery] Loading annotations: {HRBENCH_ANNOTATION_PATH}")

with open(HRBENCH_ANNOTATION_PATH, "r", encoding="utf-8") as f:
    benchmark_dataset = json.load(f)

total_available = len(benchmark_dataset)
print(f"[Dataset Ingestion] Loaded {total_available} total benchmark samples.")

# ------------------------------------------------------------------------------
# BENCHMARK EVALUATION CONFIGURATION KNOBS
# ------------------------------------------------------------------------------
NUM_EVAL_SAMPLES = 30         # Total number of samples to evaluate (e.g. 30)
START_SAMPLE_OFFSET = 0       # Starting index in dataset
CANDIDATE_POOL_SIZE = 3       # N images per sample (1 Target + N-1 Distractors)
DISTRACTOR_WINDOW = 10        # Local sampling window for realistic distractor pool
RANDOM_SEED = 42              # Master deterministic reproducibility seed

OUTPUT_DIR = "outputs/eval_multisource_n30"
VIS_DIR = os.path.join(OUTPUT_DIR, "visualizations")
TELEMETRY_DIR = os.path.join(OUTPUT_DIR, "telemetry")

os.makedirs(VIS_DIR, exist_ok=True)
os.makedirs(TELEMETRY_DIR, exist_ok=True)
print(f"[Output Directories] Initialized evaluation storage:")
print(f"  - Visual Diagnostic Cards: {VIS_DIR}")
print(f"  - Telemetry & Metrics:     {TELEMETRY_DIR}")

# Deterministic Ground-Truth Resolver
def resolve_gt_letter(answer_obj: Any, cycle_idx: int = 0) -> str:
    if isinstance(answer_obj, list):
        if 0 <= cycle_idx < len(answer_obj):
            ans = str(answer_obj[cycle_idx]).strip()
        else:
            ans = str(answer_obj[0]).strip() if answer_obj else ""
    else:
        ans = str(answer_obj).strip()
    m = re.search(r"\(?([A-Za-z])\)?", ans)
    return m.group(1).upper() if m else ans.upper()

def parse_options_text(options_raw: Any, cycle_idx: int = 0) -> Dict[str, str]:
    """Parses candidate options block into { 'A': text, 'B': text, ... }."""
    opt_dict: Dict[str, str] = {}
    if not options_raw:
        return opt_dict

    if isinstance(options_raw, list) and len(options_raw) > 0:
        if isinstance(options_raw[0], str) and ("\n" in options_raw[0] or re.match(r"^[\(\[\"'\s]*[A-Fa-f][\)\]\.\:\s]", options_raw[0].strip())):
            idx = cycle_idx if 0 <= cycle_idx < len(options_raw) else 0
            opt_block = options_raw[idx]
        elif all(isinstance(opt, str) for opt in options_raw):
            letters = ["A", "B", "C", "D", "E", "F"]
            for i, opt in enumerate(options_raw):
                let = letters[i] if i < len(letters) else str(i)
                opt_dict[let] = str(opt).strip().lower()
            return opt_dict
        else:
            opt_block = options_raw[0]
    else:
        opt_block = options_raw

    if isinstance(opt_block, dict):
        for k, v in opt_block.items():
            opt_dict[str(k).upper()] = str(v).strip().lower()
    elif isinstance(opt_block, str):
        for line in opt_block.splitlines():
            line_str = line.strip()
            if not line_str:
                continue
            m = re.match(r"^[\(\[\"'\s]*([A-Fa-f])[\)\]\.\:\s]+(.*)$", line_str)
            if m:
                opt_dict[m.group(1).upper()] = m.group(2).strip().lower()

    return opt_dict

def format_options_str(options_raw: Any, cycle_idx: int = 0) -> str:
    """Cleanly formats options for prompts: (A) text \n (B) text."""
    if isinstance(options_raw, dict):
        return chr(10).join([f"({k}) {v}" for k, v in sorted(options_raw.items())])
    opt_dict = parse_options_text(options_raw, cycle_idx=cycle_idx)
    if opt_dict:
        return chr(10).join([f"({k}) {v}" for k, v in sorted(opt_dict.items())])
    return str(options_raw) if options_raw else ""

# Build Evaluation Samples Manifest
eval_indices = list(range(START_SAMPLE_OFFSET, min(START_SAMPLE_OFFSET + NUM_EVAL_SAMPLES, total_available)))
rng = random.Random(RANDOM_SEED)

eval_manifest = []
for s_idx in eval_indices:
    sample_data = benchmark_dataset[s_idx]
    q = sample_data.get("question", "")
    c_idx = sample_data.get("cycle_category", 0)
    gt = resolve_gt_letter(sample_data.get("answer", ""), c_idx)
    raw_opts = sample_data.get("options", {})
    parsed_opts = parse_options_text(raw_opts, c_idx)
    t_rel = sample_data.get("input_image", "")
    
    # Sample Distractors
    min_w = max(0, s_idx - DISTRACTOR_WINDOW)
    max_w = min(total_available - 1, s_idx + DISTRACTOR_WINDOW)
    cand_distractors = [i for i in range(min_w, max_w + 1) if i != s_idx]
    if len(cand_distractors) < (CANDIDATE_POOL_SIZE - 1):
        cand_distractors = [i for i in range(total_available) if i != s_idx]
    
    dist_indices = rng.sample(cand_distractors, CANDIDATE_POOL_SIZE - 1)
    
    eval_manifest.append({
        "sample_index": s_idx,
        "question": q,
        "gt_letter": gt,
        "cycle_idx": c_idx,
        "options": parsed_opts,
        "raw_options": raw_opts,
        "target_path": t_rel,
        "distractor_indices": dist_indices,
        "category": sample_data.get("category", "general"),
        "question_type": "relational" if any(w in q.lower() for w in ["left", "right", "above", "below", "near", "next to", "between"]) else "attribute"
    })

print("=" * 75)
print(f"[Evaluation Manifest] Prepared {len(eval_manifest)} Evaluation Samples (Seed={RANDOM_SEED}):")
for i, m in enumerate(eval_manifest[:5]):
    opts_summary = ", ".join([f"{k}:{v[:12]}" for k, v in m["options"].items()])
    print(f"  - [{i+1:02d}/{len(eval_manifest):02d}] Sample #{m['sample_index']:03d} | GT: '{m['gt_letter']}' | Type: {m['question_type']} | Opts: [{opts_summary}] | Q: '{m['question'][:45]}...'")
if len(eval_manifest) > 5:
    print(f"  ... and {len(eval_manifest) - 5} more samples.")
print("=" * 75)


In [ ]:
# ==============================================================================
# CELL 3: MODEL LOADING (QWEN2.5-VL-7B, SIGLIP ViT-SO400M & SAM3)
# ==============================================================================
import os
import sys
import json
import torch

# Defensive path resolution for mssearch package
if "mssearch" not in sys.modules:
    for cand in [
        "/kaggle/working/MSSearch",
        "/kaggle/working",
        os.getcwd(),
        os.path.abspath("MSSearch"),
        os.path.abspath("../MSSearch"),
        os.path.abspath("../../MSSearch"),
        os.path.abspath("."),
        os.path.abspath("..")
    ]:
        if os.path.exists(os.path.join(cand, "mssearch", "__init__.py")):
            if cand not in sys.path:
                sys.path.insert(0, cand)
            break

from mssearch.models.modeling_qwenvl import ModelQwenVL
from mssearch.models.modeling_siglip import SigLIPScorer
try:
    from mssearch.CVSearch import get_cvsearch_response
except ImportError:
    try:
        from mssearch import get_cvsearch_response
    except ImportError:
        get_cvsearch_response = None
        print("[CVSearch Warning] Failed to import get_cvsearch_response")

try:
    from mssearch.models.modeling_sam3 import sam3_inference
except Exception as e:
    sam3_inference = None
    print(f"[SAM3 Warning] SAM3 import fallback: {e}")

# 1. Discover Qwen2.5-VL Weights (Local Kaggle Dataset or HF Hub ID)
QWEN_MODEL_PATH = "Qwen/Qwen2.5-VL-7B-Instruct"
if os.path.exists("/kaggle/input"):
    for root, _, files in os.walk("/kaggle/input"):
        if "config.json" in files:
            try:
                with open(os.path.join(root, "config.json"), "r", encoding="utf-8") as f:
                    c = json.load(f)
                    if c.get("model_type") in ("qwen2_5_vl", "qwen2_vl") or any("Qwen2" in a for a in c.get("architectures", [])):
                        QWEN_MODEL_PATH = root
                        print(f"[Model Discovery] Found local Qwen2.5-VL weights at: {QWEN_MODEL_PATH}")
                        break
            except Exception:
                pass

# 2. Discover SigLIP Weights (Local Kaggle Dataset or HF Hub ID)
SIGLIP_MODEL_PATH = "google/siglip-base-patch16-384"
if os.path.exists("/kaggle/input"):
    for root, _, files in os.walk("/kaggle/input"):
        if "config.json" in files:
            try:
                with open(os.path.join(root, "config.json"), "r", encoding="utf-8") as f:
                    c = json.load(f)
                    if "siglip" in c.get("model_type", "").lower() or any("Siglip" in a for a in c.get("architectures", [])):
                        SIGLIP_MODEL_PATH = root
                        print(f"[Model Discovery] Found local SigLIP weights at: {SIGLIP_MODEL_PATH}")
                        break
            except Exception:
                pass

print(f"[Model Discovery] Qwen2.5-VL Path: {QWEN_MODEL_PATH}")
print(f"[Model Discovery] SigLIP Path:     {SIGLIP_MODEL_PATH}")

# Initialize SigLIP Scorer
siglip_engine = SigLIPScorer(
    model_name=SIGLIP_MODEL_PATH,
    device=str(device_vlm)
)

# Initialize Qwen2.5-VL Engine
qwen_engine = ModelQwenVL(
    model_path=QWEN_MODEL_PATH,
    device=str(device_vlm),
    local_files_only=os.path.exists(QWEN_MODEL_PATH)
)

# Universal Helper Methods for Qwen Engine
if not hasattr(qwen_engine, "get_response"):
    def _universal_get_response(image=None, prompt=None, max_new_tokens=128, **kwargs):
        if image is not None:
            return qwen_engine.generate_response(image_pil=image, question=prompt, max_new_tokens=max_new_tokens)
        return qwen_engine.generate_text(prompt=prompt, max_new_tokens=max_new_tokens)
    qwen_engine.get_response = _universal_get_response

# 4. Load CVSearch In-Context Visual Cue Examples
cvsearch_ic_examples = {}
for ic_cand in [
    os.path.join(os.path.dirname(sys.modules["mssearch"].__file__ if "mssearch" in sys.modules else "."), "ic_examples", "hr-bench_4k.json"),
    os.path.join(os.path.dirname(sys.modules["mssearch"].__file__ if "mssearch" in sys.modules else "."), "ic_examples", "vstar.json"),
    "MSSearch/mssearch/ic_examples/hr-bench_4k.json",
    "/kaggle/working/MSSearch/mssearch/ic_examples/hr-bench_4k.json"
]:
    if os.path.exists(ic_cand):
        try:
            with open(ic_cand, "r", encoding="utf-8") as f:
                cvsearch_ic_examples = json.load(f)
            print(f"[IC Examples] Loaded {len(cvsearch_ic_examples)} in-context examples from: {ic_cand}")
            break
        except Exception:
            pass

print("=" * 75)
print("[Model Initialization] Qwen2.5-VL, SigLIP, and CVSearch engines loaded successfully.")
print("=" * 75)


In [ ]:
# ==============================================================================
# CELL 4: ALGORITHMIC ENGINES: OLD CVSSEARCH vs. NEXT-GEN REDESIGN
# ==============================================================================
from dataclasses import dataclass
from typing import Tuple, List, Dict, Any, Optional, Union
import time
import re
import json
import numpy as np
from PIL import Image

# ------------------------------------------------------------------------------
# 0. Robust Predicted Option Extractor
# ------------------------------------------------------------------------------
def extract_predicted_option(resp: Any, options: Any, cycle_idx: int = 0) -> str:
    """Extracts multiple-choice option letter ('A', 'B', 'C', 'D') from model output."""
    if resp is None:
        return "N/A"
    resp_str = str(resp).strip()
    if not resp_str or resp_str.lower() in ("none", "n/a", "null", "no answer found", "[error: image not found]"):
        return "N/A"

    # Numeric index check (0 -> A, 1 -> B, ...)
    if resp_str.isdigit():
        idx = int(resp_str)
        if 0 <= idx < 26:
            return chr(65 + idx)

    # 0. XML tag extraction <answer>...</answer>
    xml_m = re.search(r"<answer>(.*?)</answer>", resp_str, re.DOTALL | re.IGNORECASE)
    if xml_m:
        inside = xml_m.group(1).strip()
        if inside.lower() in ("none", "n/a", "null", "no answer found"):
            return "N/A"
        if inside.isdigit():
            idx = int(inside)
            if 0 <= idx < 26:
                return chr(65 + idx)
        m_let = re.search(r"\b([A-Fa-f])\b", inside)
        if m_let:
            return m_let.group(1).upper()
        m_lead = re.match(r"^[\(\[\"'\s]*([A-Fa-f])[\)\]\.\:\s]", inside)
        if m_lead:
            return m_lead.group(1).upper()
        resp_str = inside

    # 1. Exact single letter alone: 'A', '(A)', '[A]', 'A.'
    m = re.match(r"^[\s\(\[\"']*([A-Fa-f])[\)\]\"']?[\.\:\,]?\s*$", resp_str)
    if m:
        return m.group(1).upper()

    # 2. Explicit prefix like 'The answer is (A)', 'Choice B', 'Option C'
    match_prefix = re.search(
        r"(?:(?:the\s+)?(?:best\s+)?(?:final\s+)?(?:answer|choice|option)\s*(?:is|\:|\=)?\s*)[\(\[]?([A-Fa-f])[\)\]]?(?:[\.\:\,\s]|\Z)",
        resp_str,
        re.IGNORECASE,
    )
    if match_prefix:
        return match_prefix.group(1).upper()

    # 3. Leading letter with text: '(A) 27B' or 'A. 27B'
    m_lead = re.match(r"^\s*[\(\[]?([A-Fa-f])[\)\]\.\:\s]+", resp_str)
    if m_lead:
        return m_lead.group(1).upper()

    # 4. Fallback: match prediction content against option candidate text
    opt_dict = options if isinstance(options, dict) else parse_options_text(options, cycle_idx=cycle_idx)
    if opt_dict:
        r_lower = resp_str.lower().strip()
        for letter, text in opt_dict.items():
            if text:
                t_clean = str(text).lower().strip()
                if r_lower == t_clean or r_lower.startswith(t_clean) or t_clean in r_lower:
                    return letter
                if len(r_lower) >= 3 and r_lower in t_clean:
                    return letter

    return "N/A"

# ------------------------------------------------------------------------------
# 1. Direct Canvas Baseline Engine (Old Stage 2 Direct Answering)
# ------------------------------------------------------------------------------
def run_direct_canvas_baseline(
    image: Image.Image,
    question: str,
    options: Any,
    options_str: str,
    vlm_model: Any
) -> Dict[str, Any]:
    t0 = time.perf_counter()
    canvas = image.copy()
    if max(canvas.width, canvas.height) > 1024:
        canvas.thumbnail((1024, 1024), Image.Resampling.LANCZOS)
    
    prompt = f"""Question: {question}
Options:
{options_str}

Select the best option (A, B, C, or D) that answers the question based on the image.
State ONLY the option letter enclosed in <answer> tags, for example: <answer>(A)</answer>."""

    resp = vlm_model.get_response(image=canvas, prompt=prompt, max_new_tokens=128)
    latency = time.perf_counter() - t0
    pred = extract_predicted_option(resp, options)
    return {
        "pred": pred,
        "response": str(resp).strip(),
        "latency": latency,
        "input_size": (canvas.width, canvas.height),
        "regime": "DirectCanvas"
    }

# ------------------------------------------------------------------------------
# 2. Genuine Old CVSearch Baseline Engine (Hierarchical Tree + VLM Scoring)
# ------------------------------------------------------------------------------
def run_old_cvsearch_baseline(
    candidate_images: List[Image.Image],
    target_pool_idx: int,
    question: str,
    options: Union[List[str], Dict[str, str]],
    options_str: str,
    vlm_model: Any,
    sam_model: Any = None,
    ic_examples: Any = None
) -> Dict[str, Any]:
    """
    Executes the genuine previous design visual search pipeline (CVSearch).
    Performs adaptive tree bisection and VLM existence confidence scoring.
    """
    t0 = time.perf_counter()
    target_img = candidate_images[target_pool_idx]
    
    if isinstance(options, dict):
        opts_list = [f"{k}. {v}" for k, v in sorted(options.items())]
    elif isinstance(options, list):
        opts_list = [f"{chr(65+idx)}. {opt}" if not opt.strip().startswith(f"({chr(65+idx)})") and not opt.strip().startswith(f"{chr(65+idx)}.") else opt for idx, opt in enumerate(options)]
    else:
        opts_list = [str(options)]
        
    annotation = {
        "input_image": target_img,
        "question": question,
        "options": opts_list,
        "answer_type": "option_single"
    }
    
    def pop_limit_func(max_depth):
        return max(4, max_depth * 3)
    
    try:
        resp = get_cvsearch_response(
            sam_model=sam_model,
            zoom_model=vlm_model,
            nlp_model=None,
            annotation=annotation,
            ic_examples=ic_examples if ic_examples else {},
            decomposed_question_template="What is the appearance of the {}?",
            answering_confidence_threshold_lower=0.0,
            answering_confidence_threshold_upper=0.6,
            fast_threshold=0.8,
            pop_limit=pop_limit_func,
            threshold_descrease=[0.1, 0.1, 0.2],
            search_mode=True
        )
    except Exception as e:
        print(f"[CVSearch] Warning during execution: {e}")
        prompt = f"""Question: {question}
Options:
{options_str}

Select the best option from the choices above that answers the question based on the image.
State ONLY the option letter enclosed in <answer> tags, for example: <answer>(A)</answer>."""
        resp = vlm_model.get_response(image=target_img, prompt=prompt, max_new_tokens=128)
    
    latency = time.perf_counter() - t0
    
    searched_nodes = annotation.get("searched_nodes", [])
    is_pruned = False
    
    if searched_nodes and hasattr(searched_nodes[0], "state"):
        bx, by, bw, bh = searched_nodes[0].state.bbox
        bx = max(0, min(target_img.width - 1, bx))
        by = max(0, min(target_img.height - 1, by))
        bw = max(1, min(target_img.width - bx, bw))
        bh = max(1, min(target_img.height - by, bh))
        best_crop = target_img.crop((bx, by, bx + bw, by + bh))
        best_box = (bx, by, bx + bw, by + bh)
        if len(searched_nodes) == 1 and getattr(searched_nodes[0], "is_root", False) and annotation.get("search_mode") == 0:
            is_pruned = True
    else:
        best_crop = target_img.copy()
        best_box = (0, 0, target_img.width, target_img.height)
        is_pruned = True
        
    pred = extract_predicted_option(resp, options)
    
    return {
        "pred": pred,
        "response": str(resp).strip(),
        "latency": latency,
        "pruning_cliff": is_pruned,
        "best_crop": best_crop,
        "best_box": best_box,
        "regime": "OldCVSearchPipeline"
    }

# ------------------------------------------------------------------------------
# 3. Next-Gen Redesign Multi-Source Engine (Dual-Granularity Visual Search)
# ------------------------------------------------------------------------------
def run_nextgen_redesign_pipeline(
    candidate_images: List[Image.Image],
    target_pool_idx: int,
    question: str,
    options: Union[List[str], Dict[str, str]],
    options_str: str,
    siglip_model: Any,
    vlm_model: Any,
    sam_model: Any = None,
    ic_examples: Any = None
) -> Dict[str, Any]:
    """
    Next-Gen Redesign Visual Search:
    - Stage 1: In-context visual cue semantic decomposition.
    - Stage 3: Fast SigLIP spatial token saliency heatmap (single forward pass).
    - Stage 4: Continuous evidential context consolidation (Peak + Macro Envelope).
    - Stage 6: Decoupled multi-regime reading (Macro Context + Native 1:1 Detail).
    """
    t0 = time.perf_counter()
    target_img = candidate_images[target_pool_idx]
    tw, th = target_img.size
    
    # -------------------------------------------------------------------------
    # Stage 1: Dual-Granularity Semantic Decomposition
    # -------------------------------------------------------------------------
    targets = []
    if ic_examples and hasattr(vlm_model, "generate_visual_cues_using_ic"):
        try:
            targets = vlm_model.generate_visual_cues_using_ic(ic_examples, question)
        except Exception:
            targets = []

    if not targets:
        stop_words = {"what", "which", "where", "is", "are", "the", "in", "on", "at", "of", "to", "a", "an", "and", "or", "side", "color", "number", "displayed"}
        words = [w.lower() for w in re.findall(r'\b[A-Za-z]{3,}\b', question) if w.lower() not in stop_words]
        targets = words[:2] if words else ["target object"]

    dense_q = [f"a photo of {t}" for t in targets]

    # -------------------------------------------------------------------------
    # Stage 3: Fast SigLIP Unpooled Spatial Token Saliency (Single Global Pass)
    # -------------------------------------------------------------------------
    try:
        s_dense, s_pool, token_heatmap = siglip_model.extract_dense_spatial_token_saliency(
            target_img, text_queries=dense_q
        )
    except Exception as e:
        print(f"[Stage 3 SigLIP] Warning during spatial token saliency: {e}")
        token_heatmap = np.ones((24, 24), dtype=np.float32)
        s_dense, s_pool = 0.5, 0.5

    # -------------------------------------------------------------------------
    # Stage 4: Evidential Context Consolidation (Native Peak + Macro Envelope)
    # -------------------------------------------------------------------------
    hm_h, hm_w = token_heatmap.shape
    peak_y, peak_x = np.unravel_index(np.argmax(token_heatmap), token_heatmap.shape)
    cx = int((peak_x + 0.5) / hm_w * tw)
    cy = int((peak_y + 0.5) / hm_h * th)

    # 1. Native 1:1 Peak Crop (Camera Native Resolution Window: 640x640)
    pw = min(640, tw)
    ph = min(640, th)
    px1 = max(0, min(tw - pw, cx - pw // 2))
    py1 = max(0, min(th - ph, cy - ph // 2))
    peak_box = (px1, py1, px1 + pw, py1 + ph)
    peak_crop = target_img.crop(peak_box)

    # 2. Unified Macro Holistic Context Envelope (Bounds Salient Cluster with 30% Context Margin)
    thresh = 0.65 * float(np.max(token_heatmap))
    active_ys, active_xs = np.where(token_heatmap >= thresh)
    if len(active_xs) == 0:
        active_xs, active_ys = np.array([peak_x]), np.array([peak_y])

    cell_w = tw / hm_w
    cell_h = th / hm_h
    raw_mx1 = int(np.min(active_xs) * cell_w)
    raw_my1 = int(np.min(active_ys) * cell_h)
    raw_mx2 = int((np.max(active_xs) + 1) * cell_w)
    raw_my2 = int((np.max(active_ys) + 1) * cell_h)

    # Expand with 30% spatial context padding (min dimension 1280 or full canvas)
    bw_raw = raw_mx2 - raw_mx1
    bh_raw = raw_my2 - raw_my1
    min_span = min(1280, min(tw, th))
    target_bw = max(min_span, int(bw_raw * 1.4))
    target_bh = max(min_span, int(bh_raw * 1.4))
    mcx = (raw_mx1 + raw_mx2) // 2
    mcy = (raw_my1 + raw_my2) // 2

    mx1 = max(0, min(tw - target_bw, mcx - target_bw // 2))
    my1 = max(0, min(th - target_bh, mcy - target_bh // 2))
    mx2 = min(tw, max(target_bw, mx1 + target_bw))
    my2 = min(th, max(target_bh, my1 + target_bh))
    macro_box = (mx1, my1, mx2, my2)
    macro_crop = target_img.crop(macro_box)

    e_post = float(s_dense)

    # -------------------------------------------------------------------------
    # Stage 6: Decoupled Multi-Regime Reading
    # -------------------------------------------------------------------------
    reading_prompt = f"""Question: {question}
Options:
{options_str}

Select the best option (A, B, C, or D) that answers the question based on the visual evidence.
State ONLY the option letter enclosed in <answer> tags, for example: <answer>(A)</answer>."""

    # 1. Macro Holistic Context Reading
    macro_input = macro_crop.copy()
    if max(macro_input.width, macro_input.height) > 1536:
        macro_input.thumbnail((1536, 1536), Image.Resampling.LANCZOS)
    resp_macro = vlm_model.get_response(image=macro_input, prompt=reading_prompt, max_new_tokens=128)
    pred_macro = extract_predicted_option(resp_macro, options)

    # 2. Native 1:1 Peak Detail Reading
    resp_peak = vlm_model.get_response(image=peak_crop, prompt=reading_prompt, max_new_tokens=128)
    pred_peak = extract_predicted_option(resp_peak, options)

    # Unified Decision Logic:
    # - Relational questions require macro contextual field of view
    # - Fine-grained attribute reading benefits from native 1:1 peak crop
    is_relational = any(w in question.lower() for w in ["left", "right", "above", "below", "near", "next to", "between"])
    if is_relational:
        pred_unified = pred_macro if pred_macro != "N/A" else pred_peak
    else:
        pred_unified = pred_peak if pred_peak != "N/A" else pred_macro

    latency = time.perf_counter() - t0

    return {
        "pred_macro": pred_macro,
        "pred_peak": pred_peak,
        "pred_unified": pred_unified,
        "response_macro": str(resp_macro).strip(),
        "response_peak": str(resp_peak).strip(),
        "macro_crop": macro_crop,
        "peak_crop": peak_crop,
        "macro_box": macro_box,
        "peak_box": peak_box,
        "e_post": e_post,
        "latency": latency
    }


In [ ]:
# ==============================================================================
# CELL 5: DETAILED PER-SAMPLE VISUAL DIAGNOSTIC CARD GENERATOR
# ==============================================================================
import matplotlib.pyplot as plt
import matplotlib.patches as patches_mpl

def generate_sample_diagnostic_card(
    sample_meta: Dict[str, Any],
    direct_res: Dict[str, Any],
    old_res: Dict[str, Any],
    redesign_res: Dict[str, Any],
    target_img: Image.Image,
    save_path: str
):
    """
    Constructs and saves a publication-grade 4-panel visual inspection card:
      Panel 1: Candidate Full Canvas with Overlaid Bounding Boxes (CVSearch, Macro, Peak)
      Panel 2: Old CVSearch Searched Region & Predicted Output
      Panel 3: Next-Gen Redesign Macro Holistic Context Envelope (Localized Context)
      Panel 4: Next-Gen Redesign Native 1:1 Peak Detail Crop (Camera Resolution)
    """
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    gt = sample_meta["gt_letter"]
    q = sample_meta["question"]
    s_idx = sample_meta["sample_index"]
    
    # -------------------------------------------------------------------------
    # Panel 1: Full Canvas Overview with Overlays
    # -------------------------------------------------------------------------
    thumb = target_img.copy()
    thumb.thumbnail((1024, 1024), Image.Resampling.LANCZOS)
    axes[0].imshow(thumb)
    axes[0].set_title(f"1. Target Canvas (Sample #{s_idx:03d})\nOrig Size: {target_img.width}x{target_img.height}px", fontsize=11, fontweight="bold")
    axes[0].axis("off")
    
    scale_x = thumb.width / float(target_img.width)
    scale_y = thumb.height / float(target_img.height)
    
    # Overlay CVSearch box (Dodger Blue dashed)
    if old_res.get("best_box"):
        ob = old_res["best_box"]
        rect_cv = patches_mpl.Rectangle(
            (ob[0] * scale_x, ob[1] * scale_y),
            (ob[2] - ob[0]) * scale_x,
            (ob[3] - ob[1]) * scale_y,
            linewidth=2.5, edgecolor="dodgerblue", facecolor="none", linestyle="--", label="CVSearch Box"
        )
        axes[0].add_patch(rect_cv)
        
    # Overlay Redesign Macro Envelope (Lime Green solid)
    if redesign_res.get("macro_box"):
        mb = redesign_res["macro_box"]
        rect_macro = patches_mpl.Rectangle(
            (mb[0] * scale_x, mb[1] * scale_y),
            (mb[2] - mb[0]) * scale_x,
            (mb[3] - mb[1]) * scale_y,
            linewidth=2.5, edgecolor="limegreen", facecolor="none", label="Redesign Macro"
        )
        axes[0].add_patch(rect_macro)
        
    # Overlay Redesign Peak Box (Gold solid)
    if redesign_res.get("peak_box"):
        pb = redesign_res["peak_box"]
        rect_peak = patches_mpl.Rectangle(
            (pb[0] * scale_x, pb[1] * scale_y),
            (pb[2] - pb[0]) * scale_x,
            (pb[3] - pb[1]) * scale_y,
            linewidth=2.5, edgecolor="gold", facecolor="none", label="Redesign 1:1 Peak"
        )
        axes[0].add_patch(rect_peak)
        
    axes[0].legend(loc="lower left", fontsize=8, framealpha=0.8)
    
    # -------------------------------------------------------------------------
    # Panel 2: Old CVSearch Pipeline
    # -------------------------------------------------------------------------
    old_pred = old_res.get("pred", "N/A")
    old_correct = (old_pred == gt)
    old_color = "forestgreen" if old_correct else "crimson"
    
    if old_res.get("best_crop") is not None:
        axes[1].imshow(old_res["best_crop"])
        axes[1].set_title(f"2. Old CVSearch Pipeline\nPred: '{old_pred}' (GT: '{gt}') | Lat: {old_res['latency']:.2f}s",
                          fontsize=11, fontweight="bold", color=old_color)
    else:
        axes[1].text(0.5, 0.5, f"CVSearch Pruned\nPred: '{old_pred}'",
                     ha="center", va="center", fontsize=13, fontweight="bold", color="crimson")
        axes[1].set_title("2. Old CVSearch Pipeline", fontsize=11, fontweight="bold", color="crimson")
    axes[1].axis("off")
    for spine in axes[1].spines.values():
        spine.set_edgecolor(old_color)
        spine.set_linewidth(3)
        
    # -------------------------------------------------------------------------
    # Panel 3: Redesign Macro Holistic Context
    # -------------------------------------------------------------------------
    macro_pred = redesign_res.get("pred_macro", "N/A")
    macro_correct = (macro_pred == gt)
    macro_color = "forestgreen" if macro_correct else "crimson"
    axes[2].imshow(redesign_res["macro_crop"])
    axes[2].set_title(f"3. Redesign Macro Context\nPred: '{macro_pred}' (GT: '{gt}') | Post: {redesign_res['e_post']:.2f}",
                      fontsize=11, fontweight="bold", color=macro_color)
    axes[2].axis("off")
    for spine in axes[2].spines.values():
        spine.set_edgecolor(macro_color)
        spine.set_linewidth(3)
        
    # -------------------------------------------------------------------------
    # Panel 4: Redesign Native 1:1 Peak Detail
    # -------------------------------------------------------------------------
    peak_pred = redesign_res.get("pred_peak", "N/A")
    peak_correct = (peak_pred == gt)
    peak_color = "forestgreen" if peak_correct else "crimson"
    axes[3].imshow(redesign_res["peak_crop"])
    axes[3].set_title(f"4. Redesign Native 1:1 Peak Crop\nPred: '{peak_pred}' (GT: '{gt}') | Lat: {redesign_res['latency']:.2f}s",
                      fontsize=11, fontweight="bold", color=peak_color)
    axes[3].axis("off")
    for spine in axes[3].spines.values():
        spine.set_edgecolor(peak_color)
        spine.set_linewidth(3)
        
    plt.suptitle(f"Sample #{s_idx:03d} Diagnostic Comparison | Question: '{q}'\nGround Truth: Option '{gt}'",
                 fontsize=13, fontweight="bold", color="#1a237e")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()


In [ ]:
# ==============================================================================
# CELL 6: BATCH EVALUATION EXECUTION LOOP (N=30 SAMPLES)
# ==============================================================================
benchmark_results = []
checkpoint_file = os.path.join(TELEMETRY_DIR, "checkpoint_results.jsonl")

print("=" * 85)
print(f"STARTING COMPREHENSIVE MULTI-SAMPLE EVALUATION: N = {len(eval_manifest)} SAMPLES")
print("=" * 85)

for i, sample in enumerate(eval_manifest):
    s_idx = sample["sample_index"]
    q = sample["question"]
    gt = sample["gt_letter"]
    c_idx = sample.get("cycle_idx", 0)
    opts_str = format_options_str(sample["options"], cycle_idx=c_idx)
    
    t_start_sample = time.perf_counter()
    print(f"\n--- [{i+1:02d}/{len(eval_manifest):02d}] Processing Sample #{s_idx:03d} | Q: '{q[:50]}...' | GT: '{gt}' ---")
    
    # 1. Load Candidate Pool (1 Target + N-1 Distractors)
    t_rel = sample["target_path"]
    t_path = os.path.join(HRBENCH_BASE_DIR, t_rel) if not os.path.exists(t_rel) else t_rel
    target_canvas = Image.open(t_path).convert("RGB")
    
    pool_images = [target_canvas]
    for d_idx in sample["distractor_indices"]:
        d_rel = benchmark_dataset[d_idx].get("input_image", "")
        d_path = os.path.join(HRBENCH_BASE_DIR, d_rel) if not os.path.exists(d_rel) else d_rel
        pool_images.append(Image.open(d_path).convert("RGB"))
        
    # Target is placed at pool index 0 for controlled evaluation
    target_pool_idx = 0
    
    # 2. Run Direct Canvas Baseline (Old Stage 2 Direct Answering)
    res_direct = run_direct_canvas_baseline(target_canvas, q, sample["options"], opts_str, qwen_engine)
    
    # 3. Run Genuine CVSearch Baseline Pipeline (Old CVSearch)
    res_old = run_old_cvsearch_baseline(
        pool_images, target_pool_idx, q, sample["options"], opts_str, qwen_engine,
        sam_model=sam3_engine if "sam3_engine" in globals() else None,
        ic_examples=cvsearch_ic_examples if "cvsearch_ic_examples" in globals() else None
    )
    
    # 4. Run Next-Gen Redesign Multi-Source Pipeline
    res_redesign = run_nextgen_redesign_pipeline(
        pool_images, target_pool_idx, q, sample["options"], opts_str, siglip_engine, qwen_engine,
        sam_model=sam3_engine if "sam3_engine" in globals() else None,
        ic_examples=cvsearch_ic_examples if "cvsearch_ic_examples" in globals() else None
    )
    
    # Evaluation Correctness Booleans
    acc_direct = (res_direct["pred"] == gt)
    acc_old = (res_old["pred"] == gt)
    acc_macro = (res_redesign["pred_macro"] == gt)
    acc_peak = (res_redesign["pred_peak"] == gt)
    acc_unified = (res_redesign["pred_unified"] == gt)
    
    # Generate & Save Per-Sample Diagnostic Visualization Card
    vis_path = os.path.join(VIS_DIR, f"sample_{s_idx:03d}_diagnostic.png")
    generate_sample_diagnostic_card(sample, res_direct, res_old, res_redesign, target_canvas, vis_path)
    
    # Record Telemetry Metrics Record
    rec = {
        "sample_index": s_idx,
        "question": q,
        "ground_truth": gt,
        "question_type": sample["question_type"],
        "pred_direct": res_direct["pred"],
        "acc_direct": acc_direct,
        "lat_direct": res_direct["latency"],
        "pred_old": res_old["pred"],
        "acc_old": acc_old,
        "lat_old": res_old["latency"],
        "old_pruned": res_old["pruning_cliff"],
        "pred_macro": res_redesign["pred_macro"],
        "acc_macro": acc_macro,
        "pred_peak": res_redesign["pred_peak"],
        "acc_peak": acc_peak,
        "pred_unified": res_redesign["pred_unified"],
        "acc_unified": acc_unified,
        "lat_redesign": res_redesign["latency"],
        "e_post": res_redesign["e_post"],
        "vis_card_path": vis_path
    }
    benchmark_results.append(rec)
    
    # Append to streaming checkpoint file
    with open(checkpoint_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec) + "\n")
        
    # Print Live Progress 1-Liner
    print(f"  [Sample #{s_idx:03d} Result] GT: '{gt}' | Direct: '{res_direct['pred']}' ({acc_direct}) | Old CVS: '{res_old['pred']}' ({acc_old}) | Redesign Peak: '{res_redesign['pred_peak']}' ({acc_peak}) | Card Saved.")

print("\n" + "=" * 85)
print(f"EVALUATION COMPLETE: Processed {len(benchmark_results)} samples. Checkpoint: {checkpoint_file}")
print("=" * 85)


In [ ]:
# ==============================================================================
# CELL 7: GLOBAL ACCURACY SYNTHESIS & STATISTICAL REPORTING
# ==============================================================================
total_n = len(benchmark_results)

# Overall Accuracies
acc_direct_pct = sum(r["acc_direct"] for r in benchmark_results) / total_n * 100.0
acc_old_pct = sum(r["acc_old"] for r in benchmark_results) / total_n * 100.0
acc_macro_pct = sum(r["acc_macro"] for r in benchmark_results) / total_n * 100.0
acc_peak_pct = sum(r["acc_peak"] for r in benchmark_results) / total_n * 100.0
acc_unified_pct = sum(r["acc_unified"] for r in benchmark_results) / total_n * 100.0

# Pruning Cliff Statistics
pruning_cliff_rate = sum(r["old_pruned"] for r in benchmark_results) / total_n * 100.0

# Subcategory Splits
attr_samples = [r for r in benchmark_results if r["question_type"] == "attribute"]
rel_samples = [r for r in benchmark_results if r["question_type"] == "relational"]

def get_cat_acc(sample_list, key):
    return (sum(r[key] for r in sample_list) / len(sample_list) * 100.0) if sample_list else 0.0

# Latencies
avg_lat_direct = sum(r["lat_direct"] for r in benchmark_results) / total_n
avg_lat_old = sum(r["lat_old"] for r in benchmark_results) / total_n
avg_lat_redesign = sum(r["lat_redesign"] for r in benchmark_results) / total_n

print("=" * 88)
print("              COMPREHENSIVE STATISTICAL BENCHMARK REPORT (N = %d)" % total_n)
print("=" * 88)
print(f"{'REGIME / EVALUATION ENGINE':<32} | {'OVERALL ACC':<12} | {'ATTRIBUTE':<12} | {'RELATIONAL':<12} | {'AVG LATENCY':<12}")
print("-" * 88)
print(f"{'1. Direct Canvas (Old Stage 2)':<32} | {acc_direct_pct:>10.1f}% | {get_cat_acc(attr_samples, 'acc_direct'):>10.1f}% | {get_cat_acc(rel_samples, 'acc_direct'):>10.1f}% | {avg_lat_direct:>10.2f}s")
print(f"{'2. Old CVSearch (Quadtree)':<32} | {acc_old_pct:>10.1f}% | {get_cat_acc(attr_samples, 'acc_old'):>10.1f}% | {get_cat_acc(rel_samples, 'acc_old'):>10.1f}% | {avg_lat_old:>10.2f}s")
print(f"{'3. Redesign Macro Context':<32} | {acc_macro_pct:>10.1f}% | {get_cat_acc(attr_samples, 'acc_macro'):>10.1f}% | {get_cat_acc(rel_samples, 'acc_macro'):>10.1f}% | {avg_lat_redesign:>10.2f}s")
print(f"{'4. Redesign Native 1:1 Peak':<32} | {acc_peak_pct:>10.1f}% | {get_cat_acc(attr_samples, 'acc_peak'):>10.1f}% | {get_cat_acc(rel_samples, 'acc_peak'):>10.1f}% | {avg_lat_redesign:>10.2f}s")
print(f"{'5. Redesign Unified Consensus':<32} | {acc_unified_pct:>10.1f}% | {get_cat_acc(attr_samples, 'acc_unified'):>10.1f}% | {get_cat_acc(rel_samples, 'acc_unified'):>10.1f}% | {avg_lat_redesign:>10.2f}s")
print("=" * 88)
print(f"[Key Failure Diagnostic] Old CVSearch Pruning Cliff Rate (<0.25): {pruning_cliff_rate:.1f}% of samples completely dropped!")
print(f"[Key Failure Diagnostic] Next-Gen Redesign Pruning Cliff Rate:   0.0% (Zero binary dropouts).")
print("=" * 88)

# Save Master Summary JSON
summary_report = {
    "total_samples": total_n,
    "accuracies": {
        "direct_canvas": acc_direct_pct,
        "old_cvsearch_quadtree": acc_old_pct,
        "redesign_macro_context": acc_macro_pct,
        "redesign_native_peak": acc_peak_pct,
        "redesign_unified": acc_unified_pct
    },
    "category_breakdown": {
        "attribute_count": len(attr_samples),
        "relational_count": len(rel_samples),
        "attribute_acc_peak": get_cat_acc(attr_samples, "acc_peak"),
        "relational_acc_macro": get_cat_acc(rel_samples, "acc_macro"),
        "relational_acc_peak": get_cat_acc(rel_samples, "acc_peak")
    },
    "pruning_cliff_rate_old": pruning_cliff_rate,
    "avg_latencies": {
        "direct_canvas": avg_lat_direct,
        "old_cvsearch": avg_lat_old,
        "redesign": avg_lat_redesign
    },
    "samples_telemetry": benchmark_results
}

summary_json_path = os.path.join(OUTPUT_DIR, "summary_metrics.json")
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(summary_report, f, indent=2)

print(f"[Report Export] Master summary JSON saved to: {summary_json_path}")

In [ ]:
# ==============================================================================
# CELL 8: PUBLICATION-GRADE SUMMARY VISUALIZATIONS & ARTIFACT EXPORT
# ==============================================================================
# 1. Summary Accuracy Comparison Bar Chart
fig, ax = plt.subplots(figsize=(10, 6))
regimes = ["Direct Canvas", "Old CVSearch", "Redesign Macro", "Redesign 1:1 Peak", "Redesign Unified"]
accs = [acc_direct_pct, acc_old_pct, acc_macro_pct, acc_peak_pct, acc_unified_pct]
bar_colors = ["#78909c", "#e57373", "#64b5f6", "#81c784", "#2e7d32"]

bars = ax.bar(regimes, accs, color=bar_colors, edgecolor="black", linewidth=1.5, width=0.6)
ax.set_ylim(0, 105)
ax.set_ylabel("Accuracy (%)", fontsize=12, fontweight="bold")
ax.set_title(f"Multi-Regime Accuracy Comparison on HRBench (N = {total_n})", fontsize=14, fontweight="bold")

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2.0, yval + 1.5, f"{yval:.1f}%", ha="center", va="bottom", fontsize=11, fontweight="bold")

ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
acc_chart_path = os.path.join(OUTPUT_DIR, "accuracy_comparison_bar.png")
plt.savefig(acc_chart_path, dpi=200)
plt.show()

# 2. Pruning Cliff & Failure Mode Pie Chart
fig, ax = plt.subplots(figsize=(7, 5))
labels = ["Old CVSearch: Pruned at Cliff (<0.25)", "Old CVSearch: Evaluated"]
sizes = [pruning_cliff_rate, max(0.0, 100.0 - pruning_cliff_rate)]
colors = ["#d32f2f", "#4caf50"]
ax.pie(sizes, labels=labels, autopct="%1.1f%%", startangle=140, colors=colors,
       wedgeprops={"edgecolor": "black", "linewidth": 1.5})
ax.set_title("Old CVSearch Search Cliff Breakdown", fontsize=12, fontweight="bold")
plt.tight_layout()
cliff_chart_path = os.path.join(OUTPUT_DIR, "pruning_cliff_breakdown.png")
plt.savefig(cliff_chart_path, dpi=200)
plt.show()

print("=" * 75)
print("EVALUATION ARTIFACTS READY FOR INSPECTION:")
print(f"  - Summary Metrics JSON:       {summary_json_path}")
print(f"  - Accuracy Comparison Chart:  {acc_chart_path}")
print(f"  - Pruning Cliff Breakdown:    {cliff_chart_path}")
print(f"  - Per-Sample Diagnostic Cards: {VIS_DIR} (Total: {total_n} PNG files)")
print("=" * 75)